# Seminar 02: GANs

## Overview

**Generative Adversarial Networks (GANs)** are a class of generative models introduced by Ian Goodfellow and colleagues in 2014. They consist of two neural networks (a Generator and a Discriminator) trained in a min-max (adversarial) game:
- The **Generator** ($G$) aims to produce synthetic data indistinguishable from real data.
- The **Discriminator** ($D$) tries to discriminate between real and fake data.

The training objective can be summarized as:
$$
\min_G \max_D \; \mathcal{L}(G, D) = \mathbb{E}_{x \sim p_\text{data}}[\log D(x)] \;+\; \mathbb{E}_{z \sim p_z}[\log (1 - D(G(z)))]
$$

where $x$ is real data from distribution $p_\text{data}$, and $z$ is noise drawn from a prior $p_z$ (e.g., Gaussian). Over training:
- $D$ learns to give a high confidence to real samples and low confidence to generated samples.
- $G$ learns to fool $D$ by producing realistic samples.

In this seminar, we'll cover:
1. **Unconditional GAN** on MNIST zeros.
2. **Conditional GAN** on CIFAR100.
3. **f-GAN** variants with different divergences.

We'll also introduce important **engineering tips and tricks**, including:
- One-sided label smoothing ([paper](https://arxiv.org/abs/1606.03498))
- Gradient clipping
- Dropout in the Discriminator
- Decaying latent variance for inference
- Balanced update rates
- Batch Normalization in the generator

In [ ]:
# Imports & Setup
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch.nn.utils as nn_utils

# If we're on Apple MPS, just disable torch.compile
_can_compile = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    ("mps" if torch.backends.mps.is_available() else "cpu")
)
print("Device:", device)

# if device.type in ["cuda","cpu"]:
#     try:
#         torch.compile
#         _can_compile = True
#     except AttributeError:
#         print("torch.compile is NOT available in this environment.")
#         pass

# Set a fixed random seed for reproducibility
torch.manual_seed(42)

## Data Loading & Preprocessing

We'll demonstrate an unconditional GAN on **MNIST zeros**:
- We'll only load digit **"0"** examples, upscaled to 64x64.
- We'll highlight recommended engineering tips like:
  - **One-sided label smoothing** (replace label=1 with 0.9)
  - **Gradient clipping** (avoid exploding gradients)
  - **Decaying latent variance** on inference (`noise_factor`)
  - **Dropout** in the Discriminator
  - **Fully convolutional** Discriminator option
  - **Balanced G/D** update rate

In [ ]:
# Data Hyperparameters
batch_size = 128
image_size = 32

transform_mnist = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]) # map to [-1,1]
])

mnist_dataset = torchvision.datasets.MNIST(
    root="../data/MNIST",
    train=True,
    download=True,
    transform=transform_mnist
)


class OnlyZerosDataset(torch.utils.data.Dataset):
    """
    Dataset of only digit "0" from the MNIST dataset
    """
    def __init__(self, mnist_dataset):
        self.images = []
        for img, label in tqdm(mnist_dataset, desc="Filtering zeros"):
            if label == 0:
                self.images.append(img)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        return self.images[idx]

zeros_dataset = OnlyZerosDataset(mnist_dataset)
zeros_loader = DataLoader(zeros_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# A helper function for un-normalizing the image so we can visualize it properly
def process_image(img_tensor):
    """
    Un-normalize a grayscale image from [-1,1] -> [0,1] and move it to CPU for plotting.
    """
    img_tensor = img_tensor.squeeze(0).detach().cpu()
    img_tensor = (img_tensor + 1) / 2
    return img_tensor

# Quick sample visualization
plt.figure(figsize=(6, 6))
for i in range(1, 10):
    img = zeros_dataset[i]
    plt.subplot(3, 3, i)
    plt.imshow(process_image(img), cmap="gray")
    plt.axis("off")
plt.suptitle("MNIST zeros sample", fontsize=16)
plt.show()

## Unconditional GAN

In an **unconditional GAN**, we do not provide any class labels or auxiliary information. The Generator simply maps random noise to images, and the Discriminator tries to distinguish those images from real data.  

**Key steps**:
1. Sample noise $z \sim \mathcal{N}(0,I)$.  
2. Generate a fake image $G(z)$.  
3. The Discriminator sees both real and fake images, outputs real/fake scores.  
4. Update $D$ to better separate real vs. fake.  
5. Update $G$ to produce more convincing images that fool $D$.

In [4]:
class UpsampleBlock(nn.Module):
    """
    Increases the spatial resolution by factor 2 using bilinear upsampling,
    then a Conv -> BN -> LeakyReLU block repeated.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.layers(x)


class DownsampleBlock(nn.Module):
    """
    Decreases the spatial resolution by factor 2 with stride=2 convolution.
    """
    def __init__(self, in_channels, out_channels, dropout_p=0.0):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_p),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_p),
        )
    def forward(self, x):
        return self.layers(x)


class Generator(nn.Module):
    """
    Unconditional Generator for MNIST zeros:
    - We create random noise of shape (noise_channels, noise_spatial_size, noise_spatial_size).
    - 4 UpsampleBlocks to get from 4x4 to 32x32.
    - Final conv1x1 -> Tanh to produce output image in [-1,1].
    - We'll allow a 'noise_factor' for inference so we can reduce noise variance.
    """
    def __init__(self, noise_channels=4, noise_spatial_size=4, image_channels=1, base_channels=32):
        super().__init__()
        self.noise_channels = noise_channels
        self.noise_spatial_size = noise_spatial_size
        # We'll go from base_channels up to 4*base_channels in steps
        # so that each UpsampleBlock doubles the number of channels.
        # This is adjustable to get deeper G if needed.
        self.layers = nn.Sequential(
            UpsampleBlock(noise_channels, base_channels),            # 4x4 -> 8x8
            UpsampleBlock(base_channels, base_channels*2),           # 8x8 -> 16x16
            UpsampleBlock(base_channels*2, base_channels*4),         # 16x16 -> 32x32
            nn.Conv2d(base_channels*4, image_channels, kernel_size=1),
            nn.Tanh()
        )

    def forward(self, batch_size, device, noise_factor=1.0):
        # Create random latent: shape (batch_size, noise_channels, noise_spatial_size, noise_spatial_size)
        latent = torch.randn(
                    batch_size,
                    self.noise_channels,
                    self.noise_spatial_size,
                    self.noise_spatial_size,
                    device=device
                )
        latent = latent * noise_factor  # Decay latent variance on inference if needed
        return self.layers(latent)


class Discriminator(nn.Module):
    """
    Unconditional Discriminator for MNIST zeros:
    - 4 DownsampleBlocks (from 32x32 to 4x4).
    - A final "fully-convolutional" approach:
        * Instead of an fc layer, do a convolution that yields 1 channel of size 4x4,
          and then a global average to produce a single scalar for the real/fake score.
    - We also incorporate dropout in the DownsampleBlocks for better regularization.
    """
    def __init__(self, image_channels=1, base_channels=32, dropout_p=0.3):
        super().__init__()
        self.main = nn.Sequential(
            DownsampleBlock(image_channels, base_channels, dropout_p=dropout_p),       # 32x32 -> 16x16
            DownsampleBlock(base_channels, base_channels*2, dropout_p=dropout_p),      # 16x16 -> 8x8
            DownsampleBlock(base_channels*2, base_channels*4, dropout_p=dropout_p)     # 8x8 -> 4x4
        )

        self.conv_out = nn.Conv2d(base_channels*4, 1, kernel_size=4, stride=1)

    def forward(self, x):
        x = self.main(x)      # shape is (N, 128, 4, 4) if base_channels=32*4=128
        x = self.conv_out(x)  # shape (N, 1, 1, 1)
        return x.view(-1, 1)  # Flatten to (N, 1)

In [5]:
# Training Hyperparameters
lrG = 0.0001    # Generator learning rate
lrD = 0.0001    # Discriminator learning rate
num_epochs = 30
grad_clip_value = 1.0
noise_channels = 4
noise_spatial_size = 4  # We'll create noise of shape (4,4)
label_smoothing = 0.9  # one-sided label smoothing
update_ratio = 1  # how many times to update D before G

**When to pick which ratio?**  
- **If your Discriminator saturates easily** (making it hard for the Generator to learn), you might lower how often D gets updated (`update_ratio < 1`) or reduce $D$’s LR.  
- **If your Generator is struggling** (perhaps mode collapse) and $D$ can’t provide a stable gradient, you might increase how often you update $D$ (`update_ratio > 1`).  

Always keep an eye on training curves (both $D$ and $G$ losses). Too high a Discriminator update ratio can cause the Generator’s gradients to vanish. Too low a Discriminator update ratio can cause $D$ to lag and fail to provide meaningful feedback. Balancing them properly is key.

In [6]:
# Create G & D
generator = Generator(
    noise_channels=noise_channels,
    noise_spatial_size=noise_spatial_size,
    image_channels=1,
    base_channels=image_size
)

discriminator = Discriminator(
    image_channels=1,
    base_channels=image_size,
    dropout_p=0.3
)
 
# (Optional) compile with PyTorch 2.5 for speed (CUDA/CPU only)
if _can_compile:
    generator = torch.compile(generator)
    discriminator = torch.compile(discriminator)

# Send models to device
generator.to(device)
discriminator.to(device)

# Create optimizers
optim_G = optim.Adam(generator.parameters(), lr=lrG)
optim_D = optim.Adam(discriminator.parameters(), lr=lrD, betas=(0.5, 0.999))

scheduler_G = optim.lr_scheduler.StepLR(optim_G, step_size=20, gamma=0.5)
scheduler_D = optim.lr_scheduler.StepLR(optim_D, step_size=20, gamma=0.5)

losses = {"D": [], "G": []}

In [ ]:
# Training loop
for epoch in range(num_epochs):
    for real_images in tqdm(zeros_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
        real_images = real_images.to(device)
        batch_sz = real_images.size(0)

        # Determine integer steps for D and G
        if update_ratio >= 1.0:
            d_steps = int(update_ratio)
            g_steps = 1
        else:
            d_steps = 1
            g_steps = int(1.0 / update_ratio)

        # -----------------------------
        #  Train Discriminator (d_steps times)
        # -----------------------------

        for _ in range(d_steps):
            optim_D.zero_grad(set_to_none=True)

            # Real images
            real_output = discriminator(real_images)
            real_labels = torch.ones_like(real_output, device=device) * label_smoothing
            loss_real = F.binary_cross_entropy_with_logits(real_output, real_labels)

            # Fake images
            fake_images = generator(batch_sz, device, noise_factor=1.0).detach()
            fake_output = discriminator(fake_images)
            fake_labels = torch.zeros_like(fake_output, device=device)

            loss_fake = F.binary_cross_entropy_with_logits(fake_output, fake_labels)
            loss_D = (loss_real + loss_fake) * 0.5
            loss_D.backward()

            # Gradient clipping
            nn.utils.clip_grad_norm_(discriminator.parameters(), grad_clip_value)
            optim_D.step()

        # -----------------------------
        #  Train Generator (g_steps times)
        # -----------------------------
        for _ in range(g_steps):
            optim_G.zero_grad(set_to_none=True)

            gen_images = generator(batch_sz, device, noise_factor=1.0)
            gen_output = discriminator(gen_images)
            # We want them to be recognized as 'real', with smoothing
            desired_labels = torch.ones_like(gen_output) * label_smoothing

            loss_G = F.binary_cross_entropy_with_logits(gen_output, desired_labels)
            loss_G.backward()

            # Gradient clipping
            nn.utils.clip_grad_norm_(generator.parameters(), grad_clip_value)
            optim_G.step()

        # Save final losses from last step
        losses["D"].append(loss_D.item())
        losses["G"].append(loss_G.item())

    # Step the LR schedulers after each epoch
    scheduler_G.step()
    scheduler_D.step()

    print(f"Epoch [{epoch+1}/{num_epochs}] | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            "epoch": epoch+1,
            "generator_state_dict": generator.state_dict(),
            "discriminator_state_dict": discriminator.state_dict(),
            "optimizer_G_state_dict": optim_G.state_dict(),
            "optimizer_D_state_dict": optim_D.state_dict()
        }, f"weights/gan_checkpoint_epoch_{epoch+1}.pth")

        # -----------------------------
        # Print some generated zeros every 5 epochs
        # -----------------------------
        generator.eval()
        with torch.no_grad():
            sample_fakes = generator(16, device, noise_factor=1.0).cpu()
            
        fig, axs = plt.subplots(2, 8, figsize=(12, 3))
        for i in range(16):
            ax = axs[i // 8, i % 8]
            ax.imshow(process_image(sample_fakes[i]), cmap='gray')
            ax.axis("off")
        plt.suptitle(f"Generated MNIST zeros at Epoch {epoch+1}")
        plt.show()
        generator.train()

In [ ]:
# Plot loss curves
plt.figure(figsize=(8, 6))
plt.plot(losses["D"], label="Discriminator Loss")
plt.plot(losses["G"], label="Generator Loss")
plt.legend()
plt.title("Unconditional GAN on MNIST Zeros - Losses")
plt.grid()
plt.show()

In [ ]:
# Generate a small grid of images at the end
generator.eval()  # Switch to eval mode; can also reduce noise_factor here if you want
with torch.no_grad():
    # We can reduce the latent variance a bit for "cleaner" images
    final_fakes = generator(36, device, noise_factor=0.7).cpu()

plt.figure(figsize=(6, 6))
for i in range(36):
    plt.subplot(6, 6, i + 1)
    plt.imshow(process_image(final_fakes[i]), cmap='gray')
    plt.axis("off")
plt.suptitle("Generated MNIST zeros (Unconditional GAN)", fontsize=16)
plt.show()

### Common Tricks & Observations

1. **One-Sided Label Smoothing**: Helps prevent $D$ from becoming overconfident.  
2. **Random Label Flipping** (occasionally label real as fake and vice versa): This can also help stabilize training but is not always necessary.  
3. **Batch Normalization**: Helps with stable training in both $G$ and $D$.  
4. **Avoid Sparse Gradients**: For some activations or losses, gradients can vanish or saturate. The log-sigmoid approach can help.  
5. **Mode Collapse**: A common issue where $G$ finds a subset of modes that fool $D$ but fails to capture the full data diversity. Techniques like **minibatch discrimination**, **history of old generators**, or **WGAN** variants can help.  

## Conditional GAN

A **Conditional GAN (cGAN)** incorporates **class labels** or some auxiliary information into both $G$ and $D$.  

- The **Generator** receives both noise $z$ and a label $y$ as input.  
- The **Discriminator** sees both the real/fake image and the corresponding label $y$.  

This allows the GAN to learn class-specific image generation, improving control over the outputs.

In [10]:
# We'll build a config dict to keep track of parameters easily.

class DotDict(dict):
    """Helper class to allow dot notation: d.key -> d['key']"""
    __getattr__ = dict.__getitem__
    __setattr__ = dict.__setitem__

config = DotDict({
    "noise_channels": 4,
    "noise_spatial_size": 4,
    "image_size": 32,
    "image_channels": 3,
    "batch_size": 32,
    "num_attrs": 100,  # CIFAR100 has 100 classes
    "device": device,
    "max_epochs": 30,
    "grad_clip_value": 1.0,
    "lrG": 0.0002,
    "lrD": 0.0002,
    "label_smoothing": 0.9
})

In [ ]:
# Prepare CIFAR100
transform_cifar = transforms.Compose([
    transforms.Resize((config.image_size, config.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],[0.5, 0.5, 0.5])
])
cifar_dataset = torchvision.datasets.CIFAR100(
    root="../data/CIFAR100",
    train=True,
    transform=transform_cifar,
    download=True
)
cifar_loader = DataLoader(cifar_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)

In [ ]:
# Helper function for unnormalizing an image
def process_cifar_image(img_tensor):
    """
    Un-normalize from [-1, 1] to [0, 1] and permute (3, H, W) -> (H, W, 3) for plotting.
    """
    img_tensor = img_tensor.permute(1, 2, 0).detach().cpu()
    img_tensor = (img_tensor + 1) / 2
    return img_tensor

# Quick check of data
label2name_cifar = {val:key for key,val in cifar_dataset.class_to_idx.items()}
plt.figure(figsize=(6, 6))
for i in range(1, 10):
    img, lbl = cifar_dataset[i]
    plt.subplot(3, 3, i)
    plt.imshow(process_cifar_image(img))
    plt.title(label2name_cifar[lbl])
    plt.axis("off")
plt.suptitle("CIFAR100 Samples")
plt.show()

In [13]:
# Conditional Upsample/Downsample blocks

class CondUpsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.layers(x)


class CondDownsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout_p=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_p),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout_p),
        )
    def forward(self, x):
        return self.layers(x)


class CondGenerator(nn.Module):
    """
    Conditional generator:
    - We'll embed the label to produce an embedding of shape (1, noise_spatial_size, noise_spatial_size)
    - We'll concatenate that embedding with the random noise along the channel dimension
    - Then pass through upsampling blocks until we reach 64x64
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.label_embedding = nn.Embedding(config.num_attrs,
                                            config.noise_spatial_size * config.noise_spatial_size)

        # We'll keep doubling the channel count. Start from noise_channels+1
        # because we have noise + label embedding in the channel dimension.
        base_channels = config.image_size
        self.initial_channels = config.noise_channels + 1

        self.up1 = CondUpsampleBlock(self.initial_channels, base_channels)       # 4x4 -> 8x8
        self.up2 = CondUpsampleBlock(base_channels, base_channels*2)             # 8x8 -> 16x16
        self.up3 = CondUpsampleBlock(base_channels*2, base_channels*4)           # 16x16 -> 32x32
        # self.up4 = CondUpsampleBlock(base_channels*4, base_channels*4)         # 32x32 -> 64x64

        self.conv_out = nn.Conv2d(base_channels*4, self.config.image_channels, kernel_size=1)
        self.tanh = nn.Tanh()

    def forward(self, batch_size, labels, device, noise_factor=1.0):
        # Create random noise
        noise = torch.randn(batch_size, self.config.noise_channels,
                            self.config.noise_spatial_size, self.config.noise_spatial_size,
                            device=device)
        noise = noise * noise_factor

        # Create label embedding
        # shape => (batch_size, noise_spatial_size * noise_spatial_size)
        embed = self.label_embedding(labels)
        # reshape => (batch_size, 1, noise_spatial_size, noise_spatial_size)
        embed = embed.view(batch_size, 1, self.config.noise_spatial_size, self.config.noise_spatial_size)

        # concat => shape (batch_size, noise_channels+1, noise_spatial_size, noise_spatial_size)
        x = torch.cat([noise, embed], dim=1)

        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        # x = self.up4(x)

        x = self.conv_out(x)
        x = self.tanh(x)
        return x


class CondDiscriminator(nn.Module):
    """
    Conditional discriminator:
    - We'll embed the label to produce an extra channel with the same spatial resolution (config.image_size x config.image_size).
    - We'll concatenate input image + embedded label along the channel dimension.
    - Then pass through downsampling blocks until shape is 4x4.
    - Finally do a fully convolutional approach => conv to 1 dimension => global average => single real/fake score.
    """
    def __init__(self, config, dropout_p=0.3):
        super().__init__()
        self.config = config
        self.label_embedding = nn.Embedding(config.num_attrs,
                                            config.image_size * config.image_size)

        base_channels = config.image_size
        # input channels = image_channels + 1
        self.down1 = CondDownsampleBlock(config.image_channels+1, base_channels, dropout_p=dropout_p)     # config.image_size -> config.image_size / 2
        self.down2 = CondDownsampleBlock(base_channels, base_channels*2, dropout_p=dropout_p)             # config.image_size / 2 -> config.image_size / 4
        self.down3 = CondDownsampleBlock(base_channels*2, base_channels*4, dropout_p=dropout_p)           # config.image_size / 4 -> config.image_size / 8
        # self.down4 = CondDownsampleBlock(base_channels*4, base_channels*4, dropout_p=dropout_p)           # 8->4

        self.conv_out = nn.Conv2d(base_channels*4, 1, kernel_size=4, stride=1)

    def forward(self, x, labels):
        # x: shape (batch, image_channels, config.image_size, config.image_size)
        # labels: shape (batch,)
        embed = self.label_embedding(labels)  # shape (batch, config.image_size * config.image_size)
        embed = embed.view(-1, 1, self.config.image_size, self.config.image_size)  # shape (batch, 1, config.image_size, config.image_size)
        x = torch.cat([x, embed], dim=1)
        x = self.down1(x)
        x = self.down2(x)
        x = self.down3(x)
        # x = self.down4(x)
        
        x = self.conv_out(x)  # shape (batch, 1, 1, 1)
        return x.view(-1,1)


class ConditionalGAN:
    """
    Wraps up the conditional generator & discriminator,
    handles training steps, etc.
    """
    def __init__(self, config):
        self.config = config
        self.G = CondGenerator(config).to(config.device)
        self.D = CondDiscriminator(config, dropout_p=0.3).to(config.device)
        if _can_compile:
            self.G = torch.compile(self.G)
            self.D = torch.compile(self.D)

        self.optim_G = optim.Adam(self.G.parameters(), lr=config.lrG, betas=(0.5, 0.999))
        self.optim_D = optim.Adam(self.D.parameters(), lr=config.lrD, betas=(0.5, 0.999))
        # self.optim_D = optim.SGD(self.D.parameters(), lr=config.lrD)

    def train_discriminator(self, real_images, labels):
        self.optim_D.zero_grad(set_to_none=True)

        # Real forward
        real_out = self.D(real_images, labels)
        real_labels = torch.ones_like(real_out) * self.config.label_smoothing
        loss_real = F.binary_cross_entropy_with_logits(real_out, real_labels)

        # Fake forward
        fake = self.G(real_images.size(0), labels, self.config.device, noise_factor=1.0).detach()
        fake_out = self.D(fake, labels)
        fake_labels = torch.zeros_like(fake_out)
        loss_fake = F.binary_cross_entropy_with_logits(fake_out, fake_labels)

        loss_D = 0.5 * (loss_real + loss_fake)
        loss_D.backward()
        nn.utils.clip_grad_norm_(self.D.parameters(), self.config.grad_clip_value)
        self.optim_D.step()
        return loss_D.item()

    def train_generator(self, batch_size, labels):
        self.optim_G.zero_grad(set_to_none=True)

        fake = self.G(batch_size, labels, self.config.device, noise_factor=1.0)
        out = self.D(fake, labels)
        target = torch.ones_like(out) * self.config.label_smoothing
        loss_G = F.binary_cross_entropy_with_logits(out, target)

        loss_G.backward()
        nn.utils.clip_grad_norm_(self.G.parameters(), self.config.grad_clip_value)
        self.optim_G.step()
        return loss_G.item()

    @torch.no_grad()
    def sample(self, n_samples, labels=None, noise_factor=0.7):
        """
        Sample images from the generator, optionally with given labels.
        If labels is None, use random labels in [0, num_attrs).
        """
        if labels is None:
            labels = torch.randint(0, self.config.num_attrs, (n_samples,), device=self.config.device)
        fake = self.G(n_samples, labels, self.config.device, noise_factor=noise_factor)
        return fake.cpu()

In [ ]:
# Instantiate and train
cond_gan = ConditionalGAN(config)
losses_cond = {"D": [], "G": []}

print("\n--- Training Conditional GAN on CIFAR100 ---\n")
for epoch in range(config.max_epochs):
    for real_images, labels in tqdm(cifar_loader, desc=f"Epoch {epoch+1}/{config.max_epochs}", leave=False):
        real_images = real_images.to(config.device)
        labels = labels.to(config.device)

        d_loss = cond_gan.train_discriminator(real_images, labels)
        g_loss = cond_gan.train_generator(real_images.size(0), labels)

        losses_cond["D"].append(d_loss)
        losses_cond["G"].append(g_loss)

    print(f"Epoch [{epoch+1}/{config.max_epochs}] | D Loss: {d_loss:.4f} | G Loss: {g_loss:.4f}")

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            "epoch": epoch+1,
            "generator_state_dict": cond_gan.G.state_dict(),
            "discriminator_state_dict": cond_gan.D.state_dict(),
            "optimizer_G_state_dict": cond_gan.optim_G.state_dict(),
            "optimizer_D_state_dict": cond_gan.optim_D.state_dict()
        }, f"weights/cgan_checkpoint_epoch_{epoch+1}.pth")

        # -----------------------------
        # Print some generated zeros every 5 epochs
        # -----------------------------
        with torch.no_grad():
            sample_labels = torch.randint(0, config.num_attrs, (16,), device=config.device)
            samples = cond_gan.sample(16, labels=sample_labels, noise_factor=1.0)
            
        fig, axs = plt.subplots(2, 8, figsize=(12, 3))
        for i in range(16):
            ax = axs[i // 8, i % 8]
            ax.imshow(process_cifar_image(samples[i]), cmap='gray')
            ax.axis("off")
        plt.suptitle(f"Generated Imagres by CGAN on CIFAR100 at Epoch {epoch+1}")
        plt.show()

In [ ]:
# Visualize losses
plt.figure(figsize=(8, 6))
plt.plot(losses_cond["D"], label="Discriminator Loss")
plt.plot(losses_cond["G"], label="Generator Loss")
plt.title("Conditional GAN on CIFAR100 - Losses")
plt.legend()
plt.show()

In [ ]:
# Sample and visualize
cond_gan.G.eval()
with torch.no_grad():
    # Generate 36 random labels and images
    sample_labels = torch.randint(0, config.num_attrs, (36,), device=config.device)
    samples = cond_gan.sample(36, labels=sample_labels, noise_factor=0.7)

plt.figure(figsize=(6, 6))
for i in range(36):
    plt.subplot(6, 6, i+1)
    plt.imshow(process_cifar_image(samples[i]))
    plt.axis("off")
plt.suptitle("Generated Images (Conditional GAN, CIFAR100)", fontsize=16)
plt.show()

### Observations & Advantages of cGAN
- **Label control**: We can direct the generator to produce images of a specific class.  
- **Improved sample variety**: Conditioned on the label, the generator focuses on class-specific features.  
- **Potential for multi-modal generation**: Different classes can lead to diverse generation.

## f-GAN

**f-GAN** generalizes the GAN training objective by choosing different **f-divergences** (e.g., KLD, JS, Reverse KL, etc.). The objective modifies how $D$ is trained and how the **Generator** updates.  

- $g$ and $f^*$ come from the **Fenchel conjugate**.  
- For certain choices (like "GAN"), we recover the original min-max game.  
- For others (like "JSD", "KLD"), we get different theoretical properties and potentially different training behaviors.

In [31]:
class Activation_g(nn.Module):
    """
    Activation function g(v) in f-GANs, controlling the divergence type.
    For 'GAN' divergence, it's -log(1+exp(-v)) which is log(sigmoid(v)).
    """
    def __init__(self, divergence="GAN"):
        super().__init__()
        self.divergence = divergence

    def forward(self, v):
        if self.divergence == "KLD":
            return v
        elif self.divergence == "RKL":
            return -torch.exp(-v)
        elif self.divergence == "CHI":
            return v
        elif self.divergence == "SQH":
            return 1 - torch.exp(-v)
        elif self.divergence == "JSD":
            # log(2) - log(1+exp(-v)) = log(2) - softplus(-v)
            return torch.log(torch.tensor(2.0, device=v.device)) - torch.log(1.0 + torch.exp(-v))
        elif self.divergence == "GAN":
            # -log(1 + exp(-v)) = log(sigmoid(v))
            return -torch.log(1.0 + torch.exp(-v))


class Conjugate_f(nn.Module):
    """
    Conjugate function f*(t) for different divergences.
    """
    def __init__(self, divergence="GAN"):
        super().__init__()
        self.divergence = divergence

    def forward(self, t):
        if self.divergence == "KLD":
            return torch.exp(t - 1)
        elif self.divergence == "RKL":
            return -1 - torch.log(-t)
        elif self.divergence == "CHI":
            return 0.25 * t**2 + t
        elif self.divergence == "SQH":
            return t / (1.0 - t)
        elif self.divergence == "JSD":
            # -log(2 - exp(t))
            return -torch.log(2.0 - torch.exp(t))
        elif self.divergence == "GAN":
            # -log(1 - exp(t))
            return -torch.log(1.0 - torch.exp(t))


class VLoss(nn.Module):
    """
    V function in f-GAN: E_p_data[g(V(x))]
    """
    def __init__(self, divergence="GAN"):
        super().__init__()
        self.activation = Activation_g(divergence)

    def forward(self, v):
        return torch.mean(self.activation(v))


class QLoss(nn.Module):
    """
    Q function in f-GAN: E_p_fake[-f*(g(V(x)))]
    """
    def __init__(self, divergence="GAN"):
        super().__init__()
        self.conjugate = Conjugate_f(divergence)
        self.activation = Activation_g(divergence)

    def forward(self, v):
        return torch.mean(-self.conjugate(self.activation(v)))

We'll re-use the same **Generator** and **Discriminator** from our unconditional setup. We'll pick a certain divergence (e.g. `"SQH"`) and see how it compares.

In [ ]:
fgan_divergence = "JSD"  # e.g. "GAN", "KLD", "JSD", "RKL", "SQH", "CHI"

# We re-instantiate G, D with the unconditional architecture from above, using the updated approach
fG = Generator(noise_channels=4, noise_spatial_size=4, image_channels=1, base_channels=image_size).to(device)
fD = Discriminator(image_channels=1, base_channels=image_size, dropout_p=0.3).to(device)

if _can_compile:
    fG = torch.compile(fG)
    fD = torch.compile(fD)

Q_criterion = QLoss(fgan_divergence)
V_criterion = VLoss(fgan_divergence)

Q_optimizer = optim.Adam(fG.parameters(), lr=lrG)
V_optimizer = optim.Adam(fG.parameters(), lr=lrD, betas=(0.5, 0.999))

num_epochs_fgan = 100

In [ ]:
# Training
for ep in range(num_epochs_fgan):
    for images in tqdm(zeros_loader, desc=f"f-GAN Epoch {ep+1}/{num_epochs_fgan}", leave=False):
        images = images.to(device)
        b_size = images.size(0)

        #-------------------------
        # Train V
        #-------------------------
        fD.zero_grad(set_to_none=True)
        v_real = fD(images)  # shape (b_size, 1)

        # VLoss => E_p_data[g(V(x))], we want to maximize => minimize negative
        loss_real = -V_criterion(v_real)
        loss_real.backward(retain_graph=True)

        # Fake
        fake = fG(b_size, device, noise_factor=1.0).detach()
        v_fake = fD(fake)
        # QLoss => E_p_fake[-f*(g(V(x)))] => we want to maximize => minimize negative
        loss_fake = -Q_criterion(v_fake)
        loss_fake.backward()

        loss_V = -(loss_real + loss_fake)  # for logging
        V_optimizer.step()

        #-------------------------
        # Train G
        #-------------------------
        fG.zero_grad(set_to_none=True)
        fake_forG = fG(b_size, device, noise_factor=1.0)
        v_fake2 = fD(fake_forG)
        # Option 1: QLoss => minimize it
        # Option 2: maximize V => -VLoss
        # For demonstration, let's do the "maximize V trick" => minimize (-V_criterion)
        loss_G = -V_criterion(v_fake2)
        # Or: loss_G = Q_criterion(v_fake2)
        loss_G.backward()
        Q_optimizer.step()

    print(f"Epoch [{ep+1}/{num_epochs_fgan}] => Loss V: {loss_V.item():.4f} | Loss G: {loss_G.item():.4f}")

In [ ]:
# Visualize final f-GAN generation results
fG.eval()
with torch.no_grad():
    final_fake_fgan = fG(36, device, noise_factor=0.7).cpu()

plt.figure(figsize=(6, 6))
for i in range(36):
    plt.subplot(6, 6, i+1)
    plt.imshow(process_image(final_fake_fgan[i]), cmap='gray')
    plt.axis("off")
plt.suptitle("Generated MNIST zeros (f-GAN variant)", fontsize=16)
plt.show()

## Important Points

**1) How do we choose the input size (latent dimension) for the generator?**  
- In most GAN architectures (e.g., [DCGAN](https://arxiv.org/pdf/1511.06434)), you often see a 1D latent vector of size 100, or a small spatial latent shape like (4,4) with multiple channels.  
- **Theoretical**: There's no strict rule for the "best" latent dimension. However, if it's too small, the model may not have enough capacity to capture the complexity of the data. If it's too large, training may become unstable or produce meaningless variations.  
- **Practical**: People often pick 100–256 as the latent dimension (for 1D noise) or a small 2D shape (like 4x4) with a certain number of channels. For images up to 64×64, a 4×4 latent shape is common because each upsampling layer doubles spatial resolution (4→8→16→32→64).  

**2) Upsampling approaches: Transposed Convolution vs. `nn.Upsample`**  
- **Transposed Convolution** (a.k.a. deconvolution) is learned upsampling. It can cause [checkerboard artifacts](https://distill.pub/2016/deconv-checkerboard/) if kernel sizes/strides are not aligned properly.  
- **`nn.Upsample` + Conv2d**: Upsample with a deterministic interpolation (e.g. bilinear), then apply a normal convolution. This can reduce [checkerboard artifacts](https://distill.pub/2016/deconv-checkerboard/) because upsampling is not learned. Then the convolution refines details.  
- **Which is better?**  
  - If you want more control and fewer artifacts, using `nn.Upsample` + a follow-up convolution often yields smoother results.  
  - Transposed convolutions may produce sharper edges but can cause periodic artifacts if not carefully designed.  

## Comparisons & Final Discussion

**GAN Variants**:
- **Vanilla GAN**: Uses a Jensen-Shannon-like objective.  
- **WGAN**: Uses Earth-Mover (Wasserstein) distance for training stability, but requires weight clipping or gradient penalty.  
- **LSGAN**: Uses least-squares loss to help vanishing gradient issues.  
- **f-GAN**: A general framework that recovers different divergences by choosing $g$ and $f^*$.  
- **Conditional GAN**: Incorporates labels or attributes into G and D for more control.  

**Key Challenges**:
- **Mode Collapse**: $G$ might produce a limited variety of samples. Solutions: gradient penalties, unrolled GANs, or multi-discriminator approaches.  
- **Stability**: The adversarial training can be unstable. Tricks like label smoothing, balanced learning rates, and spectral normalization can help.  
- **Evaluation Metrics**: Inception Score, FID, and KID measure sample quality, but they are not perfect.  

**Practical Tips**:
- Start with a smaller architecture or dataset to confirm correctness.  
- If $D$ quickly saturates, reduce its capacity or LR. If $G$ saturates, try lowering LR or use label smoothing.  
- Check intermediate samples frequently to spot mode collapse or training failures early.  

## GANs: Pros & Cons

**Pros**

- Can utilize power of back-prop.
- No explicit intractable integral.
- No MCMC needed.

**Cons**
- Unclear stopping criteria.
- No explicit representation of $g_{\theta}(x)$.
- Hard to train.
- No evaluation metric so hard to compare with other models.
- Easy to get trapped in local optima that memorize training data.
- Hard to invert generative model to get back latent $z$ from generated $x$.